In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys, os
# This is not super pretty, but I think this is the best way to import stuff from ../../../util?
CODE_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../data_generation_pipeline"))
if CODE_ROOT not in sys.path:
    sys.path.insert(1, CODE_ROOT)

from spectra_stitching import load_forest_spectra, hash_spectra_args

In [ ]:
def add_noise_to_spectrum(spec, snr, mask=None, fill_value=np.nan, rng=None):

    if rng is None:
        rng = np.random  # legacy global state, seeded with np.random.seed

    if mask is None:
        valid = np.ones(spec.shape, dtype=bool)
    else:
        valid = mask
        assert valid.shape == spec.shape, f"shape of mask with {valid.shape=} doesnt match shape of spectrum with {spec.shape=}"

    sigma = np.abs(spec[valid] / snr[valid])

    noisy_spec = np.full(spec.shape, fill_value, dtype=float)
    noisy_spec[valid] = spec[valid] + rng.normal(0.0, sigma)
    noisy_spec[valid & (noisy_spec < 0)] = 0

    return noisy_spec



def load_spectra_cache(path):
    """Load a spectra cache saved by build_and_save_spectra. Returns a dict of arrays."""
    with np.load(path) as npz:
        return {
            "wavelength": npz["wavelength"],
            "flux": npz["flux"],
            "mask": npz["mask"],
            "snr": npz["snr"],
            "redshift": npz["redshift"],
            "pmf": npz["pmf"],
            "valid_spectrum": npz["valid_spectrum"],
            "z_min": float(npz["z_min"]),
            "z_max": float(npz["z_max"]),
        }


def find_closest_index(array, value):
    return int(np.argmin(np.abs(array - value)))

In [ ]:
out_file_path = "/pfs/10/project/bw21g005/ly_alpha_sbi_paper/SDSS_spectra/SDSS_support_files/spectra_cache.npz"

data = load_spectra_cache(out_file_path)

In [ ]:
data["snr"].shape

In [ ]:
wavelength_range = (3800, 4500)
n_specs = 10000
redshifts_to_use = [2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0]

args_hash = hash_spectra_args(wavelength_range, redshifts_to_use, None, n_specs, 1215.67, 123)

sim_spec_path = "/pfs/10/project/bw21g005/ly_alpha_sbi_paper/L50n512_suite/gridpoint0/lya_forest_spectra/" + f"forest_spectra_{args_hash}.hdf5"

print(sim_spec_path)

In [ ]:
wave_sim, flux_sim, usage_records = load_forest_spectra(sim_spec_path)

In [ ]:
# enforce redshift of quasar to be greater than right edge of interval

min_req_z = 4500/1215.67 - 1
print(min_req_z)

mask = (data["redshift"] >= min_req_z) & (data["valid_spectrum"] == 1)

wave_sdss = data["wavelength"]
flux_sdss = data["flux"][mask, :]
snr_sdss = data["snr"][mask, :]
mask_sdss = data["mask"][mask, :]
redshift_sdss = data["redshift"][mask]

print(flux_sdss.shape, snr_sdss.shape, mask_sdss.shape)

In [ ]:
# cut spectra to interval of simulated spectra

i0 = find_closest_index(wave_sdss, wave_sim[0])
i1 = find_closest_index(wave_sdss, wave_sim[-1])

diff = abs(wave_sdss[i0:i1].shape[0] - wave_sim.shape[0])

if 1 >= diff > 0:
    i1 += diff
else:
    raise ValueError(f"Length mismatch between SDSS and simulation spectra: {wave_sdss[i0:i1].shape[0]} vs {wave_sim.shape[0]}")


print(snr_sdss.shape)
wave_sdss = wave_sdss[i0:i1]
snr_sdss = snr_sdss[:, i0:i1]
mask_sdss = mask_sdss[:, i0:i1]
print(snr_sdss.shape)

In [ ]:
# enforce min of min_valid_pixels

min_valid_pixels = 100

mask_valid_pixels = np.sum(mask_sdss, axis=1) >= min_valid_pixels

wave_sdss = wave_sdss
snr_sdss = snr_sdss[mask_valid_pixels, :]
mask_sdss = mask_sdss[mask_valid_pixels, :]
redshift_sdss = redshift_sdss[mask_valid_pixels]
print(snr_sdss.shape)

In [ ]:
# Coverage map coloured by redshift: rows sorted by redshift, masked pixels painted black.
order = np.argsort(redshift_sdss)
z_sorted = redshift_sdss[order]
mask_sorted = mask_sdss[order]

# Good pixels carry their spectrum's redshift, masked pixels are NaN -> drawn black
z_image = np.where(mask_sorted, z_sorted[:, None], np.nan)

cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("black")  # mask_sdss == 0

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(
    z_image,
    cmap=cmap,
    vmin=z_sorted.min(),
    vmax=z_sorted.max(),
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    extent=[wave_sdss[0], wave_sdss[-1], 0, z_image.shape[0]],
)
# Observed Lyman-beta wavelength for each spectrum's redshift
LYB_REST = 1025.7223  # Angstrom
lyb_obs = LYB_REST * (1.0 + z_sorted)
ax.plot(lyb_obs, np.arange(z_image.shape[0]), color="red", lw=1.0, label=r"Ly$\beta$")
ax.set_xlim(wave_sdss[0], wave_sdss[-1])
ax.legend(loc="upper left", labelcolor="white")

ax.set_xlabel("Wavelength [$\\AA$]")
ax.set_ylabel("# of Spectrum")
fig.colorbar(im, ax=ax, label="Redshift")

plt.show()

In [ ]:
snr_mean_wave = np.nanmean(snr_sdss, axis=0)
snr_std_wave = np.nanstd(snr_sdss, axis=0)
snr_mean_spec = np.nanmean(snr_sdss, axis=1)

snr_median_wave = np.nanmedian(snr_sdss, axis=0)

snr_range = (0, 15)

# layout="none" disables the constrained/tight layout engine, which would
# otherwise ignore wspace and keep a gap between the two panels
fig, (ax, ax_hist) = plt.subplots(
    1, 2, figsize=(10, 4), sharey=True, layout="none",
    gridspec_kw={"width_ratios": [3, 1]},
)

ax.plot(wave_sdss, snr_mean_wave, color="blue", label="Mean")
ax.fill_between(
    wave_sdss,
    snr_mean_wave - snr_std_wave,
    snr_mean_wave + snr_std_wave,
    color="blue",
    alpha=0.3,
    label="1 $\sigma$"
)
ax.plot(wave_sdss, snr_median_wave, color="blue", linestyle="-.", label="Median", alpha=0.6)
ax.set_xlim(wave_sdss[0], wave_sdss[-1])
ax.set_ylim(*snr_range)
ax.set_xlabel("Wavelength [$\\AA$]")
ax.set_ylabel("SNR")
ax.legend()

ax_hist.hist(
    snr_mean_spec,
    bins=25,
    range=snr_range,
    orientation="horizontal",
    color="blue",
    alpha=0.7,
)
ax_hist.set_xlabel("# of Spectra")
ax_hist.tick_params(axis="y", labelleft=False, left=False)
ax_hist.xaxis.set_major_locator(plt.MaxNLocator(3, prune="lower"))

fig.subplots_adjust(wspace=0)
plt.show()

In [ ]:
# draw one SDSS noise realisation (snr + mask) per simulated spectrum

rng = np.random.default_rng(42)

n_sdss, n_sim = snr_sdss.shape[0], flux_sim.shape[0]
replace = n_sdss < n_sim  # only reuse SDSS spectra if there are not enough of them

sample_idx = rng.choice(n_sdss, size=n_sim, replace=replace)

snr_appl = snr_sdss[sample_idx]
mask_appl = mask_sdss[sample_idx]

assert snr_appl.shape == mask_appl.shape == flux_sim.shape, \
    f"shapes {snr_appl.shape=} / {mask_appl.shape=} dont match {flux_sim.shape=}"

noisy_flux_sim = add_noise_to_spectrum(flux_sim, snr_appl, mask_appl, rng=rng)

print(f"{flux_sim.shape} -> {noisy_flux_sim.shape}, drawn from {n_sdss} SDSS spectra with {replace=}")
print(f"masked pixel fraction: {1 - mask_appl.mean():.3f}, nan fraction in output: {np.isnan(noisy_flux_sim).mean():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
i = 5
plt.plot(wave_sim, noisy_flux_sim[i], color="orange", label="Simulated + SDSS noise + mask")
plt.plot(wave_sim, flux_sim[i], color="black", label="Simulated", alpha=0.5)
plt.xlabel("Wavelength [$\\AA$]")
plt.ylabel("Flux")
plt.ylim(0, max(2, np.nanmax(noisy_flux_sim[i])))
plt.legend()